In [1]:
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time
import datetime
import re
import os
from unidecode import unidecode
pd.options.display.max_columns = 0
pd.options.display.max_rows = 30
pd.options.display.max_colwidth = 200

In [2]:
import pandas as pd

In [117]:
df = pd.read_csv('Historical_Data.csv')

C:\Users\jtmck\AppData\Local\Temp\ipykernel_158228\3876305402.py:1: DtypeWarning: Columns (0,12,24,58,59,60,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Historical_Data.csv')


In [118]:
df.shape

(263473, 65)

In [119]:
df = df[['player', 'starter',
       'game_gameday','game_loc',
       'team_code', 'opponent_code', 'min',
       'pts', 'fgm', 'fga', 'threefm', 'threefa', 'ftm', 'fta', 'reb', 'ast',
       'stl', 'blk', 'oreb', 'to', 'pf', 'fgpct', 'twofgpct', 'ftpct', 'dreb',
       'usgpct', 'eff', 'perfscore', 'perfscoreseasonavg',
       'perfscoreseasonavgdev', 'presencerate', 'adjpresencerate', 'teamposs',
       'teamfga', 'teamfta', 'teamto',
       'threefgpct']]

In [120]:
df['starter'] = df['starter'].apply(lambda x: 1 if x == 'Y' else 0)

In [121]:
df['game_gameday'] = pd.to_datetime(df['game_gameday'])
# Sort by player and date
df = df.sort_values(by=['player', 'game_gameday'])
# Calculate the cumulative average
df['average_points'] = df.groupby('player')['pts'].expanding().mean().reset_index(level=0, drop=True)
df['average_rebounds'] = df.groupby('player')['reb'].expanding().mean().reset_index(level=0, drop=True)
df['average_assists'] = df.groupby('player')['ast'].expanding().mean().reset_index(level=0, drop=True)
df['std_dev_points'] = df.groupby('player')['pts'].expanding().std().reset_index(level=0, drop=True)
df['std_dev_rebounds'] = df.groupby('player')['reb'].expanding().std().reset_index(level=0, drop=True)
df['std_dev_assists'] = df.groupby('player')['ast'].expanding().std().reset_index(level=0, drop=True)

In [122]:
date_ranges = [
    (pd.to_datetime('2014-10-01'), pd.to_datetime('2015-09-01'), '2014'),
    (pd.to_datetime('2015-10-01'), pd.to_datetime('2016-09-01'), '2015'),
    (pd.to_datetime('2016-10-01'), pd.to_datetime('2017-09-01'), '2016'),
    (pd.to_datetime('2017-10-01'), pd.to_datetime('2018-09-01'), '2017'),
    (pd.to_datetime('2018-10-01'), pd.to_datetime('2019-09-01'), '2018'),
    (pd.to_datetime('2019-10-01'), pd.to_datetime('2020-11-01'), '2019'),
    (pd.to_datetime('2020-12-01'), pd.to_datetime('2021-09-01'), '2020'),
    (pd.to_datetime('2021-10-01'), pd.to_datetime('2022-09-01'), '2021'),
    (pd.to_datetime('2022-10-01'), pd.to_datetime('2023-09-01'), '2022'),
    (pd.to_datetime('2023-10-01'), pd.to_datetime('2024-09-01'), '2023'),
    (pd.to_datetime('2024-10-01'), pd.to_datetime('2025-09-01'), '2024'),
    (pd.to_datetime('2025-10-01'), pd.to_datetime('2026-09-01'), '2025'),
    (pd.to_datetime('2026-10-01'), pd.to_datetime('2027-09-01'), '2026')]

# Function to assign values based on date range
def assign_range(date):
    for start, end, value in date_ranges:
        if start <= date <= end:
            return value
    return 'Out of range'  # Optional: default value for dates not in any range

# Apply the function
df['season_year'] = df['game_gameday'].apply(assign_range).reset_index(drop=True)

In [123]:
df = df.sort_values(by=['player', 'season_year', 'game_gameday']).reset_index(drop=True)
df['season_avg_pts_before_game'] = (
    df.groupby(['player', 'season_year'])['pts']
    .apply(lambda x: x.expanding().mean().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)
df['season_avg_reb_before_game'] = (
    df.groupby(['player', 'season_year'])['reb']
    .apply(lambda x: x.expanding().mean().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)
df['season_avg_ast_before_game'] = (
    df.groupby(['player', 'season_year'])['ast']
    .apply(lambda x: x.expanding().mean().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)
df['season_std_pts_before_game'] = (
    df.groupby(['player', 'season_year'])['pts']
    .apply(lambda x: x.expanding().std().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)
df['season_std_reb_before_game'] = (
    df.groupby(['player', 'season_year'])['reb']
    .apply(lambda x: x.expanding().std().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)
df['season_std_ast_before_game'] = (
    df.groupby(['player', 'season_year'])['ast']
    .apply(lambda x: x.expanding().std().shift())  # Expanding mean and shift to exclude the current game
).reset_index(drop=True)

In [124]:
df = df[~df['player'].isna()]

In [125]:
rework_df = df.copy()

In [126]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
rework_df['player'] = le.fit_transform(rework_df['player'])
rework_df['game_gameday'] = le.fit_transform(rework_df['game_gameday'])
rework_df['game_loc'] = le.fit_transform(rework_df['game_loc'])
rework_df['team_code'] = le.fit_transform(rework_df['team_code'])
rework_df['opponent_code'] = le.fit_transform(rework_df['opponent_code'])

In [127]:
rework_df

,player,starter,game_gameday,game_loc,team_code,opponent_code,min,pts,fgm,fga,threefm,threefa,ftm,fta,reb,ast,stl,blk,oreb,to,pf,fgpct,twofgpct,ftpct,dreb,usgpct,eff,perfscore,perfscoreseasonavg,perfscoreseasonavgdev,presencerate,adjpresencerate,teamposs,teamfga,teamfta,teamto,threefgpct,average_points,average_rebounds,average_assists,std_dev_points,std_dev_rebounds,std_dev_assists,season_year,season_avg_pts_before_game,season_avg_reb_before_game,season_avg_ast_before_game,season_std_pts_before_game,season_std_reb_before_game,season_std_ast_before_game
0,0,0,360,1,7,9,6,9.0,3.0,5.0,1.0,1.0,2.0,2.0,3.0,0.0,0,0,2.0,0.0,1.0,60.00,50.00,100.0,1.0,38.18,7.2,4.0,0.95,3.05,6.06,7.02,92.0,86.0,28.0,7.0,100.0,4.500000,1.500000,0.500000,6.363961,2.121320,0.707107,2015,NaN,NaN,NaN,NaN,NaN,NaN
1,0,0,398,1,7,7,5,5.0,2.0,3.0,1.0,2.0,0.0,0.0,1.0,0.0,0,0,1.0,2.0,0.0,66.67,100.00,NaN,NaN,31.15,2.0,2.0,0.95,1.05,4.42,3.83,92.0,81.0,9.0,12.0,50.0,1.571429,1.428571,0.142857,2.680823,1.398586,0.363137,2015,9.0,3.0,0.0,NaN,NaN,NaN
2,0,0,499,0,7,26,21,0.0,0.0,2.0,0.0,0.0,0.0,0.0,7.0,1.0,0,1,2.0,2.0,3.0,NaN,NaN,NaN,5.0,6.40,NaN,2.0,0.95,1.05,9.71,6.60,95.0,88.0,8.0,9.0,NaN,1.941176,2.000000,0.176471,3.051036,2.291288,0.392953,2015,7.0,2.0,0.0,2.828427,1.414214,0.0
3,0,0,382,0,7,4,2,2.0,1.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0,0.0,0.0,0.0,50.00,50.00,NaN,1.0,44.61,0.9,NaN,0.95,NaN,1.45,1.69,91.0,87.0,18.0,10.0,NaN,1.571429,1.285714,0.285714,3.359422,0.951190,0.487950,2016,NaN,NaN,NaN,NaN,NaN,NaN
4,0,0,395,1,7,28,3,3.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0,0,0.0,1.0,3.0,100.00,NaN,NaN,1.0,18.14,0.8,1.0,0.95,0.05,3.85,2.42,85.0,77.0,21.0,6.0,100.0,1.416667,1.333333,0.166667,2.678478,1.435481,0.389249,2016,2.0,1.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263255,1497,0,990,1,19,15,12,2.0,1.0,3.0,0.0,0.0,0.0,0.0,3.0,0.0,0,0,0.0,1.0,3.0,33.33,33.33,NaN,3.0,12.47,NaN,1.0,1.75,NaN,6.69,5.11,94.0,81.0,6.0,17.0,NaN,2.000000,3.000000,0.000000,NaN,NaN,NaN,2014,NaN,NaN,NaN,NaN,NaN,NaN
263256,1497,0,991,0,19,9,5,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,8.45,NaN,NaN,1.75,NaN,NaN,NaN,100.0,89.0,20.0,13.0,NaN,1.000000,1.500000,0.000000,1.414214,2.121320,0.000000,2016,NaN,NaN,NaN,NaN,NaN,NaN
263257,1497,0,1427,0,29,5,5,0.0,0.0,3.0,0.0,2.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,30.11,NaN,NaN,NaN,NaN,NaN,NaN,93.0,85.0,11.0,7.0,NaN,2.400000,1.800000,0.400000,3.286335,1.643168,0.547723,2018,NaN,NaN,NaN,NaN,NaN,NaN
263258,1497,0,1113,1,19,21,20,8.0,4.0,4.0,0.0,0.0,0.0,0.0,3.0,1.0,1,0,2.0,1.0,5.0,100.00,100.00,NaN,1.0,9.30,5.6,4.0,1.75,2.25,7.08,5.67,103.0,87.0,27.0,13.0,NaN,3.000000,2.250000,0.500000,3.464102,1.500000,0.577350,2020,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
for i in rework_df.columns:
    rework_df[i] = rework_df[i].fillna(0)

In [129]:
from sklearn.model_selection import train_test_split
X = rework_df.drop(columns=['min',
       'pts', 'fgm', 'fga', 'threefm', 'threefa', 'ftm', 'fta', 'reb', 'ast',
       'stl', 'blk', 'oreb', 'to', 'pf', 'fgpct', 'twofgpct', 'ftpct', 'dreb',
       'usgpct', 'eff', 'perfscore', 'perfscoreseasonavg',
       'perfscoreseasonavgdev', 'presencerate', 'adjpresencerate', 'teamposs',
       'teamfga', 'teamfta', 'teamto',
       'threefgpct'])
y = rework_df[['pts','reb','ast']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [130]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [131]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MAE: 2.501637481323924
R²: 0.5329589451770835


In [132]:
y_pred

array([[13.66,  8.61,  2.16],
       [ 4.81,  4.14,  0.6 ],
       [ 1.65,  1.47,  0.3 ],
       ...,
       [10.87,  1.95,  1.85],
       [17.58,  3.85,  0.92],
       [ 9.73,  4.33,  6.7 ]])

In [113]:
X

,player,starter,game_gameday,game_loc,team_code,opponent_code,average_points,average_rebounds,average_assists,season_year,season_avg_pts_before_game,season_avg_reb_before_game,season_avg_ast_before_game
0,0,0,360,1,7,9,4.500000,1.500000,0.500000,2015,0.0,0.0,0.0
1,0,0,398,1,7,7,1.571429,1.428571,0.142857,2015,9.0,3.0,0.0
2,0,0,499,0,7,26,1.941176,2.000000,0.176471,2015,7.0,2.0,0.0
3,0,0,382,0,7,4,1.571429,1.285714,0.285714,2016,0.0,0.0,0.0
4,0,0,395,1,7,28,1.416667,1.333333,0.166667,2016,2.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
263255,1497,0,990,1,19,15,2.000000,3.000000,0.000000,2014,0.0,0.0,0.0
263256,1497,0,991,0,19,9,1.000000,1.500000,0.000000,2016,0.0,0.0,0.0
263257,1497,0,1427,0,29,5,2.400000,1.800000,0.400000,2018,0.0,0.0,0.0
263258,1497,0,1113,1,19,21,3.000000,2.250000,0.500000,2020,0.0,0.0,0.0


# Run Neural Network

In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [14]:
tf.keras

AttributeError: module 'tensorflow' has no attribute 'keras'

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout

In [ ]:
pip instal

In [47]:
nn = df.copy()

In [48]:
from sklearn.preprocessing import LabelEncoder
player_encoder = LabelEncoder()
nn['player'] = player_encoder.fit_transform(nn['player'])
day_encoder = LabelEncoder()
nn['game_gameday'] = day_encoder.fit_transform(nn['game_gameday'])
loc_encoder = LabelEncoder()
nn['game_loc'] = loc_encoder.fit_transform(nn['game_loc'])
team_encoder = LabelEncoder()
nn['team_code'] = team_encoder.fit_transform(nn['team_code'])
opp_encoder = LabelEncoder()
nn['opponent_code'] = opp_encoder.fit_transform(nn['opponent_code'])

In [49]:
X_numeric = df[['average_points','average_rebounds','average_assists']].values
X_categorical = df[['player', 'game_gameday','game_loc','team_code','opponent_code']].values
y = df['pts'].values

In [50]:
numeric_input = Input(shape=(X_numeric.shape[1],), name='numeric_input')

NameError: name 'Input' is not defined